# 03 · Generate and STORE a 5-minute CML IDW grid

**Depends on 01 + 02.** This fills the previous Figure 7 workflow gap. It reads the stage-01 final Graf rainfall and the released QC geometry, matches **structural CML IDs**, and calls the unchanged original midpoint-IDW functions. The grid is saved with `R(time,y,x)` in mm/h, the schema expected by the original spatial map renderer in notebook 04.

The historical grid used a different earlier 1-minute Blettner input. This is a transparent **5-minute final-product adaptation**, not an assertion that the historic grid is byte-identical. Control the extent/time window to avoid accidental multi-gigabyte grids. The default covers the July 2021 manuscript panels plus the 1-hour lookback.

**About this notebook.** Needs: the stage 01 rainfall and the preprocessed file for link geometry. Parameters: grid resolution, neighbour count, power, maximum distance, time window. Makes: a 5 min rainfall grid NetCDF using the unchanged original inverse distance weighting functions. Cost: minutes. Fidelity: adapted workflow; the historical manuscript grid came from an older 1 min input. Details: `docs/NOTEBOOK_GUIDE.md`.

In [4]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [5]:
from essd_cz.connected import grid_final_rain
project.get('rain_timeline')    # ensure notebook 02 was visited
START_GRID='2021-07-13 15:05:00'  # first 1-hour panel requires previous 12 x 5-min steps
END_GRID  ='2021-07-13 23:05:00'
RESOLUTION_DEG = 0.02          # 0.01 original gridding parameter; use 0.02 for fast DEMO
NEAREST_LINKS = 15            # original nnear
IDW_POWER = 2.0               # original p
MAX_DISTANCE_DEG = 0.3        # original max link-to-grid distance
PAD_DEG = 0.05
MAP_EXTENT = None              # or [lon_min,lon_max,lat_min,lat_max]
GRID = project.stage_dir('03_gridded')/'reviewer_final_graf_IDW_5min.nc'
if GRID.is_file():
    print('Grid already exists; refusing an unintentional overwrite:',GRID)
else:
    grid_final_rain(project.get('rainfall'),project.paths['qc'],GRID,START_GRID,END_GRID,
       resolution=RESOLUTION_DEG,nnear=NEAREST_LINKS,power=IDW_POWER,
       max_dist=MAX_DISTANCE_DEG,pad=PAD_DEG,extent=MAP_EXTENT)
project.register('spatial_grid',GRID,'adapted final-5min Graf midpoint IDW using exact original IDW functions',
  resolution=RESOLUTION_DEG,nnear=NEAREST_LINKS,power=IDW_POWER,max_dist=MAX_DISTANCE_DEG)

Grid already exists; refusing an unintentional overwrite: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/03_gridded/reviewer_final_graf_IDW_5min.nc
[REGISTERED] spatial_grid: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/03_gridded/reviewer_final_graf_IDW_5min.nc
  source: adapted final-5min Graf midpoint IDW using exact original IDW functions


PosixPath('/home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/03_gridded/reviewer_final_graf_IDW_5min.nc')

In [6]:
import xarray as xr
with xr.open_dataset(project.get('spatial_grid')) as ds:
    print('Grid dimensions:',dict(ds.sizes))
    print('Variables:',list(ds.data_vars))
    print('First and last:',ds.time.values[0], ds.time.values[-1])

Grid dimensions: {'time': 97, 'y': 127, 'x': 310}
Variables: ['R', 'nvalid']
First and last: 2021-07-13T15:05:00.000000000 2021-07-13T23:05:00.000000000
